In [ ]:
#Install Dependencies
!pip install -q unsloth
!pip install -q transformers datasets peft trl bitsandbytes accelerate evaluate rouge_score wandb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.3/70.3 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.2/447.2 kB 37.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 45.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 50.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 124.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 42.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.2/395.2 kB 38.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 80.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 135.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.6/182.6 kB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 58.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 115.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [ ]:
#Imports
import unsloth
import wandb
from unsloth import FastLanguageModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch
import os
from google.colab import userdata
from huggingface_hub import login

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [ ]:
#Connect APIs to session
login(token=userdata.get('Fine-Tuning'))
wandb.login(key=userdata.get('WANDB_API_KEY'))

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


True

In [ ]:
#mount storage
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/clinical-llm'
os.makedirs(PROJECT_DIR, exist_ok=True)

!pip install -q transformers datasets peft trl bitsandbytes \
             accelerate evaluate rouge_score wandb unsloth

#Verify GPU
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Mounted at /content/drive
True
NVIDIA L4
VRAM: 23.7 GB


In [ ]:
from datasets import load_dataset

#Load medQA dataset
dataset = load_dataset("medalpaca/medical_meadow_medqa")

print(dataset)
print("\n---Sample Entry---")
print(dataset["train"][0])

#Format Sample to return a dict for model input
def format_sample(sample):
    return {"text": f"""<|system|> You are a clinical medical assistant, answer the following question.
<|user|> {sample['instruction']}\n{sample['input']}
<|assistant|> {sample['output']}"""}

#Split dataset into train and evals
subset = dataset['train'].select(range(2500))
split = subset.train_test_split(test_size=0.1,seed=42)

print(split['train'])
print(split['test'])

format_train = split['train'].map(format_sample)
format_test = split['test'].map(format_sample)

README.md: 0.00B [00:00, ?B/s]

medical_meadow_medqa.json:   0%|          | 0.00/10.7M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/10178 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input', 'instruction', 'output'],
        num_rows: 10178
    })
})

---Sample Entry---
{'input': "Q:A 23-year-old pregnant woman at 22 weeks gestation presents with burning upon urination. She states it started 1 day ago and has been worsening despite drinking more water and taking cranberry extract. She otherwise feels well and is followed by a doctor for her pregnancy. Her temperature is 97.7°F (36.5°C), blood pressure is 122/77 mmHg, pulse is 80/min, respirations are 19/min, and oxygen saturation is 98% on room air. Physical exam is notable for an absence of costovertebral angle tenderness and a gravid uterus. Which of the following is the best treatment for this patient?? \n{'A': 'Ampicillin', 'B': 'Ceftriaxone', 'C': 'Ciprofloxacin', 'D': 'Doxycycline', 'E': 'Nitrofurantoin'},", 'instruction': 'Please answer with one of the option in the bracket', 'output': 'E: Nitrofurantoin'}
Dataset({
    features: ['input', 'instruction',

Map:   0%|          | 0/2250 [00:00<?, ? examples/s]

Map:   0%|          | 0/250 [00:00<?, ? examples/s]

In [ ]:
print(format_test[0])

{'input': "Q:A simple experiment is performed to measure the breakdown of sucrose into glucose and fructose by a gut enzyme that catalyzes this reaction. A glucose meter is used to follow the breakdown of sucrose into glucose. When no enzyme is added to the sucrose solution, the glucose meter will have a reading of 0 mg/dL; but when the enzyme is added, the glucose meter will start to show readings indicative of glucose being formed. Which of the following diabetic pharmacological agents, when added before the addition of the gut enzyme to the sucrose solution, will maintain a reading of 0 mg/dL?? \n{'A': 'Insulin', 'B': 'Glyburide', 'C': 'Metformin', 'D': 'Acarbose', 'E': 'Exenatide'},", 'instruction': 'Please answer with one of the option in the bracket', 'output': 'D: Acarbose', 'text': "<|system|> You are a clinical medical assistant, answer the following question.\n<|user|> Please answer with one of the option in the bracket\nQ:A simple experiment is performed to measure the break

In [ ]:
#Training Config
LLM_Config = BitsAndBytesConfig(
    load_in_4bit=True,
    compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

In [ ]:
#L oad the model and tokenizer
# model name --> "meta-llama/Llama-3.2-3B-Instruct"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "meta-llama/Llama-3.2-3B-Instruct",
    max_seq_length = 2048,
    load_in_4bit = True,
)

#Checking Vram available on colab
print(f"VRAM used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

==((====))==  Unsloth 2026.3.4: Fast Llama patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3.2-3b-instruct-unsloth-bnb-4bit as a legacy tokenizer.


VRAM used: 2.40 GB


In [ ]:
#Initiate Model Instance
model = FastLanguageModel.get_peft_model(
    model=model,
    r = 16,
    lora_alpha = 32,
    target_modules = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj"
],
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth"
)
#Sanity check for if QVK layers and included in trainable params
model.print_trainable_parameters()

Unsloth 2026.3.4 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


trainable params: 24,313,856 || all params: 3,237,063,680 || trainable%: 0.7511


In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

#Connect to Weights and Biases for experimental Tracking
wandb.init(project="clinical-llm", name="ablation-r16",reinit=True)

args = TrainingArguments(
    output_dir = "/content/drive/MyDrive/clinical-llm/Outputs",
    num_train_epochs = 3,
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 8,
    warmup_steps = 50,
    learning_rate = 2e-4,
    logging_steps = 10,
    save_steps = 50,
    eval_strategy = "no",
    fp16 = True,
    report_to = "wandb",
    save_total_limit = 3,
)

wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


wandb: Detected [huggingface_hub.inference, openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai/


In [ ]:
#Fine tuning trainer instance
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=format_train,
    eval_dataset=format_test,
    dataset_text_field="text",
    args=args,
    max_seq_length=2048

)

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/2250 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/250 [00:00<?, ? examples/s]

In [ ]:
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,250 | Num Epochs = 3 | Total steps = 423
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)


Step,Training Loss
10,2.110155
20,1.678807
30,1.312147
40,1.228647
50,1.196666
60,1.202021
70,1.189710
80,1.163309
90,1.149054
100,1.178936


train/epoch,▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇███
train/global_step,▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇███
train/grad_norm,█▄▂▃▂▂▁▁▁▁▁▁▁▁▁▁▂▂▂▂▃▂▂▂▂▂▃▂▃▃▃▃▄▄▄▄▃▄▃▄
train/learning_rate,▂▄▅▇████▇▇▇▇▇▆▆▆▆▆▅▅▅▅▄▄▄▄▄▄▃▃▃▃▃▂▂▂▂▂▁▁
train/loss,█▅▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁
total_flos,3.595545640753152e+16
train/epoch,3
train/global_step,423
train/grad_norm,0.61226
train/learning_rate,0.0
train/loss,0.98127


TrainOutput(global_step=423, training_loss=1.101312770347505, metrics={'train_runtime': 2396.1685, 'train_samples_per_second': 2.817, 'train_steps_per_second': 0.177, 'total_flos': 3.595545640753152e+16, 'train_loss': 1.101312770347505, 'epoch': 3.0})

In [ ]:
model.save_pretrained("/content/drive/MyDrive/clinical-llm/checkpoints/r16")
tokenizer.save_pretrained("/content/drive/MyDrive/clinical-llm/checkpoints/r16")

('/content/drive/MyDrive/clinical-llm/checkpoints/r16/tokenizer_config.json',
 '/content/drive/MyDrive/clinical-llm/checkpoints/r16/chat_template.jinja',
 '/content/drive/MyDrive/clinical-llm/checkpoints/r16/tokenizer.json')

In [ ]:
from evaluate import load
import torch
from tqdm import tqdm
rouge = load("rouge")

def evaluate_model(model, tokenizer, val_dataset):
    correct = 0
    predictions = []
    references = []
    FastLanguageModel.for_inference(model)
    for sample in tqdm(val_dataset):
        # format prompt WITHOUT the answer
        prompt = f"""<|system|> You are a clinical medical assistant. Answer the following question.
<|user|> {sample['instruction']}\n{sample['input']}
<|assistant|>"""

        # tokenize and move to GPU
        inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

        # generate response
        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=30)

        # decode output
        response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

        # check if correct answer is in response
        if sample['output'] in response:
            correct += 1

        predictions.append(response)
        references.append(sample['output'])

    accuracy = correct / len(val_dataset)
    rouge_scores = rouge.compute(predictions=predictions, references=references)

    return accuracy, rouge_scores['rougeL']

In [ ]:
checkpoints = {
    "r4":  "/content/drive/MyDrive/clinical-llm/checkpoints/r4",
    "r8":  "/content/drive/MyDrive/clinical-llm/checkpoints/r8",
    "r16": "/content/drive/MyDrive/clinical-llm/checkpoints/r16",
    "r32": "/content/drive/MyDrive/clinical-llm/checkpoints/r32",
}

results = {}

for rank, path in checkpoints.items():
    # 1. load model from checkpoint
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = path,
        max_seq_length = 2048,
        load_in_4bit = True,
)
    # 2. run evaluate_model()
    accuracy, rouge_score = evaluate_model(model, tokenizer, format_test)
    # 3. store results
    results[rank] = (accuracy, rouge_score)
    del model
    torch.cuda.empty_cache() #delete cache to free memory for the next model
    # inside loop - print current result
    print(f"Rank {rank}: Accuracy={accuracy:.3f}, ROUGE-L={rouge_score:.3f}")

# outside loop - print full comparison table
print("\n--- Final Results ---")
for rank, (acc, rouge) in results.items():
    print(f"{rank}: Accuracy={acc:.3f} | ROUGE-L={rouge:.3f}")

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "/content/drive/MyDrive/clinical-llm/checkpoints/r32",
    max_seq_length = 2048,
    load_in_4bit = True,
)

model.save_pretrained_merged(
    "/content/drive/MyDrive/clinical-llm/Merged-Model/r32",
    tokenizer,
    save_method = "merged_16bit",
)

==((====))==  Unsloth 2026.3.4: Fast Llama patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3.2-3b-instruct-unsloth-bnb-4bit as a legacy tokenizer.


config.json:   0%|          | 0.00/890 [00:00<?, ?B/s]

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


...

model-00002-of-00002.safetensors:   0%|          | 0.00/1.46G [00:00<?, ?B/s]


Unsloth: Preparing safetensor model files: 100%|██████████| 2/2 [01:39<00:00, 49.98s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)



Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [03:52<00:00, 116.19s/it]


Unsloth: Merge process complete. Saved to `/content/drive/MyDrive/clinical-llm/Merged-Model/r32`


In [ ]:
#Quantization Benchmarking
import time

MERGED_PATH = "/content/drive/MyDrive/clinical-llm/Merged-Model/r32"

quant_configs = {
    "4bit":  {"load_in_4bit": True, "load_in_8bit": False, "load_in_16bit": False},
    "fp16":  {"load_in_4bit": False, "load_in_8bit": False, "load_in_16bit": True},
}

bench_results = {}

for quant_name, quant_kwargs in quant_configs.items():
    # 1. load merged model with current quant config
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = MERGED_PATH,
        max_seq_length = 2048,
        **quant_kwargs
    )
    FastLanguageModel.for_inference(model)

    latencies = []

    for i in range(20):
        sample = format_test[i]
        prompt = f"""<|system|> You are a clinical medical assistant, answer the following question.
                <|user|> {sample['instruction']}\n{sample['input']}
                <|assistant|>"""

        inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

        start = time.time()
        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=30)
        end = time.time()

        latencies.append(end-start)

    avg_latency = sum(latencies) / len(latencies)
    # run evaluate_model() for rouge + accuracy
    accuracy, rouge_score = evaluate_model(model, tokenizer, format_test)

    bench_results[quant_name] = {
        "avg_latency_sec": avg_latency,
        "accuracy": accuracy,
        "rougeL": rouge_score
    }

    del model
    torch.cuda.empty_cache()
    print(f"{quant_name}: Latency={avg_latency:.3f}s | Accuracy={accuracy:.3f} | ROUGE-L={rouge_score:.3f}")

In [ ]:
import json

save_path = "/content/drive/MyDrive/clinical-llm/quantization_benchmark_results.json"

with open(save_path, "w") as f:
    json.dump(bench_results, f, indent=4)

print(f"Saved to {save_path}")

Saved to /content/drive/MyDrive/clinical-llm/quantization_benchmark_results.json


In [ ]:
from huggingface_hub import HfApi
api = HfApi()
api.create_repo(repo_id="aakthepaak/clinical-llm-r32", repo_type="model")

RepoUrl('https://huggingface.co/aakthepaak/clinical-llm-r32', endpoint='https://huggingface.co', repo_type='model', repo_id='aakthepaak/clinical-llm-r32')

In [ ]:
api.upload_folder(repo_id="aakthepaak/clinical-llm-r32",repo_type="model",folder_path="/content/drive/MyDrive/clinical-llm/Merged-Model/r32")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...0002-of-00002.safetensors:   0%|          |  609kB / 1.46GB            

  ...0001-of-00002.safetensors:   0%|          | 23.9MB / 4.97GB            

  ...-Model/r32/tokenizer.json: 100%|##########| 17.2MB / 17.2MB            

CommitInfo(commit_url='https://huggingface.co/aakthepaak/clinical-llm-r32/commit/7db428d050afe403e6340332cc072db2019325c2', commit_message='Upload folder using huggingface_hub', commit_description='', oid='7db428d050afe403e6340332cc072db2019325c2', pr_url=None, repo_url=RepoUrl('https://huggingface.co/aakthepaak/clinical-llm-r32', endpoint='https://huggingface.co', repo_type='model', repo_id='aakthepaak/clinical-llm-r32'), pr_revision=None, pr_num=None)